This Jupyter notebook uses dash to perform the visualization. It performs a join between the incidents and the county lookup table to allow the data to be viewed by District and Year. There are 12 KYTC Districts shown on the graph from west to east, numerically.

In [1]:
# Import modules
import os
import sqlite3
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

An interesting feature of the dash app is the ability to change the graph to display each year in the dataset by selecting the desired year by a dropdown list at the top of the graph.

In [2]:
# Initialize the Dash app
app = dash.Dash(__name__)

# Define the layout of the app
app.layout = html.Div([
    html.H1("Collision Incidents by KYTC District and Year"),
    dcc.Dropdown(
        id='year-dropdown',
        options=[{'label': str(year), 'value': str(year)} for year in range(2020, 2025)],  # Example range of years
        value='2020',  # Default value
        clearable=False
    ),
    dcc.Graph(id='incident-bar-chart')
])

In [3]:
# Define a function to fetch data and create the Plotly figure
def get_figure(selected_year):
    # Connect to the SQLite database
    conn = sqlite3.connect(os.path.join(os.getcwd(), 'data/crash_data.db'))

    # Define the query
    query = f'''
    SELECT t1.IncidentID, t2.KYTC_District_Number AS District, t2.D_District,
        t2.Cnty_Name_PC, strftime("%Y", t1.CollisionDate) AS CollisionYear,
        COUNT(*) AS IncidentCount
    FROM ksp_incidents t1
    JOIN county_district_lut t2
        ON t1.County = t2.Cnty_Name_UC
    WHERE strftime("%Y", t1.CollisionDate) = '{selected_year}'
    GROUP BY
        District
    ORDER BY
        District;
    '''

    # Execute the query and fetch the results into a DataFrame
    df = pd.read_sql_query(query, conn)
    print(df)

    # Close the database connection
    conn.close()

    # Create the bar chart using Plotly Express
    #fig = px.bar(df, x='District', y='IncidentCount', color='District',
    #             labels={'IncidentCount': 'Number of Incidents'},
    #             title=f'Number of Construction Work Zone Incidents by KYTC District for {selected_year}')

    #return fig

     # Define the color sequence you want to use
    color_sequence = px.colors.qualitative.Vivid_r
    #color_sequence = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']


    # Create the bar chart using Plotly Express
    fig = px.bar(df, x='District', y='IncidentCount', color='District',
                 color_discrete_sequence=color_sequence,
                 labels={'IncidentCount': 'Number of Incidents'},
                 title=f'Number of Incidents by District for {selected_year}')

    return fig

In [4]:
# Define the callback to update the graph
@app.callback(
    Output('incident-bar-chart', 'figure'),
    [Input('year-dropdown', 'value')]
)
def update_graph(selected_year):
    return get_figure(selected_year)

In [5]:
# Run the app on a different port (e.g., 8051)
if __name__ == '__main__':
    app.run_server(debug=True, port=8051)

    IncidentID  District     D_DISTRICT Cnty_Name_PC CollisionYear  \
0     27655143         1        Paducah     Calloway          2020   
1     27576317         2   Madisonville    Henderson          2020   
2     27009091         3  Bowling Green       Warren          2020   
3     27562611         4  Elizabethtown   Washington          2020   
4     27611802         5     Louisville    Jefferson          2020   
5     27626519         6      Covington        Boone          2020   
6     27598728         7      Lexington      Fayette          2020   
7     27443850         8       Somerset      Clinton          2020   
8     27525047         9   Flemingsburg         Boyd          2020   
9     27494394        10        Jackson        Perry          2020   
10    27633830        11     Manchester       Laurel          2020   
11    27471219        12      Pikeville        Floyd          2020   

    IncidentCount  
0              63  
1              49  
2               4  
3        